In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

pd.set_option("display.max_columns", None)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
archivos = files.upload()

Saving 02_ingresos_reales_actual.csv to 02_ingresos_reales_actual.csv


In [3]:
nombre_archivo = next(iter(archivos))

columnas = [
    "Fecha",
    "Hora",
    "Sede",
    "CantidadIngresos"
]

df_real = pd.read_csv(
    nombre_archivo,
    sep=";",
    header=None,
    names=columnas,
    encoding="utf-8-sig"
)

print("Archivo cargado:", nombre_archivo)
print("Dimensiones:", df_real.shape)

df_real.head(10)

Archivo cargado: 02_ingresos_reales_actual.csv
Dimensiones: (1029, 4)


,Fecha,Hora,Sede,CantidadIngresos
0,2026-04-10,12,Central,2
1,2026-04-10,13,Central,2
2,2026-04-10,15,Tarma,1
3,2026-04-29,14,Central,3
4,2026-05-04,15,Central,1
5,2026-05-05,11,Central,8
6,2026-05-05,12,Central,8
7,2026-05-05,17,Central,4
8,2026-05-09,0,Central,4
9,2026-05-10,22,Central,1


In [4]:
df_real["Fecha"] = pd.to_datetime(
    df_real["Fecha"],
    errors="coerce"
)

df_real["Hora"] = pd.to_numeric(
    df_real["Hora"],
    errors="coerce"
)

df_real["CantidadIngresos"] = pd.to_numeric(
    df_real["CantidadIngresos"],
    errors="coerce"
)

df_real["Sede"] = (
    df_real["Sede"]
    .astype(str)
    .str.strip()
    .str.title()
)

print(df_real.dtypes)

Fecha               datetime64[ns]
Hora                         int64
Sede                        object
CantidadIngresos             int64
dtype: object


In [5]:
print("RESUMEN DEL DATASET REAL")
print("-" * 40)

print("Filas agrupadas:", len(df_real))
print("Total de ingresos:", df_real["CantidadIngresos"].sum())
print("Fecha inicial:", df_real["Fecha"].min())
print("Fecha final:", df_real["Fecha"].max())
print("Días con registros:", df_real["Fecha"].nunique())

print("\nValores nulos:")
print(df_real.isnull().sum())

print("\nIngresos por sede:")
print(
    df_real.groupby("Sede")["CantidadIngresos"]
    .sum()
    .sort_values(ascending=False)
)

RESUMEN DEL DATASET REAL
----------------------------------------
Filas agrupadas: 1029
Total de ingresos: 32788
Fecha inicial: 2026-04-10 00:00:00
Fecha final: 2026-09-28 00:00:00
Días con registros: 98

Valores nulos:
Fecha               0
Hora                0
Sede                0
CantidadIngresos    0
dtype: int64

Ingresos por sede:
Sede
Central    32787
Tarma          1
Name: CantidadIngresos, dtype: int64


In [6]:
assert len(df_real) == 1029, "La cantidad de filas no coincide"
assert df_real["CantidadIngresos"].sum() == 32788, \
    "El total de ingresos no coincide"
assert df_real["Fecha"].notna().all(), \
    "Existen fechas inválidas"
assert df_real["Hora"].notna().all(), \
    "Existen horas inválidas"
assert df_real["CantidadIngresos"].notna().all(), \
    "Existen cantidades inválidas"

print("Validación completada correctamente")

Validación completada correctamente


In [7]:
print("Filas duplicadas:",
      df_real.duplicated(
          subset=["Fecha", "Hora", "Sede"]
      ).sum())

fuera_horario = df_real[
    ~df_real["Hora"].between(8, 20)
].copy()

print("\nRegistros agrupados fuera de 08:00 a 20:00:",
      len(fuera_horario))

print("Cantidad de ingresos fuera del horario:",
      fuera_horario["CantidadIngresos"].sum())

fuera_horario

Filas duplicadas: 0

Registros agrupados fuera de 08:00 a 20:00: 4
Cantidad de ingresos fuera del horario: 8


,Fecha,Hora,Sede,CantidadIngresos
8,2026-05-09,0,Central,4
9,2026-05-10,22,Central,1
49,2026-05-15,7,Central,2
175,2026-05-29,7,Central,1


In [8]:
df_real["NombreDia"] = df_real["Fecha"].dt.day_name()

orden_dias = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

resumen_dias = (
    df_real.groupby("NombreDia")
    .agg(
        Filas=("CantidadIngresos", "size"),
        Ingresos=("CantidadIngresos", "sum"),
        DiasRegistrados=("Fecha", "nunique")
    )
    .reindex(orden_dias)
    .fillna(0)
)

resumen_dias

,Filas,Ingresos,DiasRegistrados
NombreDia,,,
Monday,204,6478,19
Tuesday,221,7737,20
Wednesday,213,7592,20
Thursday,202,6717,18
Friday,187,4259,19
Saturday,1,4,1
Sunday,1,1,1


In [9]:
df_central = df_real[
    (df_real["Sede"] == "Central") &
    (df_real["Hora"].between(8, 20)) &
    (df_real["Fecha"].dt.dayofweek < 5)
].copy()

df_central = (
    df_central
    .groupby(["Fecha", "Hora"], as_index=False)
    ["CantidadIngresos"]
    .sum()
)

df_central["Sede"] = "Central"

df_central = df_central[
    ["Fecha", "Hora", "Sede", "CantidadIngresos"]
]

print("Filas utilizables:", len(df_central))
print("Ingresos utilizables:",
      df_central["CantidadIngresos"].sum())
print("Días utilizables:",
      df_central["Fecha"].nunique())

df_central.head(10)

Filas utilizables: 1024
Ingresos utilizables: 32779
Días utilizables: 96


,Fecha,Hora,Sede,CantidadIngresos
0,2026-04-10,12,Central,2
1,2026-04-10,13,Central,2
2,2026-04-29,14,Central,3
3,2026-05-04,15,Central,1
4,2026-05-05,11,Central,8
5,2026-05-05,12,Central,8
6,2026-05-05,17,Central,4
7,2026-05-12,8,Central,15
8,2026-05-12,9,Central,32
9,2026-05-12,10,Central,34


In [10]:
horas_biblioteca = list(range(8, 21))

fechas_activas = sorted(
    df_central["Fecha"].dt.normalize().unique()
)

grilla_horaria = pd.MultiIndex.from_product(
    [fechas_activas, horas_biblioteca],
    names=["Fecha", "Hora"]
).to_frame(index=False)

df_preparado = grilla_horaria.merge(
    df_central[["Fecha", "Hora", "CantidadIngresos"]],
    on=["Fecha", "Hora"],
    how="left"
)

df_preparado["CantidadIngresos"] = (
    df_preparado["CantidadIngresos"]
    .fillna(0)
    .astype(int)
)

df_preparado["Sede"] = "Central"

df_preparado = df_preparado[
    ["Fecha", "Hora", "Sede", "CantidadIngresos"]
]

print("Filas originales utilizables:", len(df_central))
print("Filas después de completar:", len(df_preparado))
print("Horas agregadas con cero:",
      (df_preparado["CantidadIngresos"] == 0).sum())
print("Total de ingresos conservado:",
      df_preparado["CantidadIngresos"].sum())

df_preparado.head(20)

Filas originales utilizables: 1024
Filas después de completar: 1248
Horas agregadas con cero: 224
Total de ingresos conservado: 32779


,Fecha,Hora,Sede,CantidadIngresos
0,2026-04-10,8,Central,0
1,2026-04-10,9,Central,0
2,2026-04-10,10,Central,0
3,2026-04-10,11,Central,0
4,2026-04-10,12,Central,2
5,2026-04-10,13,Central,2
6,2026-04-10,14,Central,0
7,2026-04-10,15,Central,0
8,2026-04-10,16,Central,0
9,2026-04-10,17,Central,0


In [11]:
df_preparado["DiaSemana"] = (
    df_preparado["Fecha"].dt.dayofweek
)

df_preparado["Mes"] = (
    df_preparado["Fecha"].dt.month
)

df_preparado["DiaMes"] = (
    df_preparado["Fecha"].dt.day
)

df_preparado["SemanaAnio"] = (
    df_preparado["Fecha"]
    .dt.isocalendar()
    .week
    .astype(int)
)

df_preparado["EsLunes"] = (
    df_preparado["DiaSemana"] == 0
).astype(int)

df_preparado["EsViernes"] = (
    df_preparado["DiaSemana"] == 4
).astype(int)

df_preparado["Turno"] = pd.cut(
    df_preparado["Hora"],
    bins=[7, 12, 17, 20],
    labels=["Mañana", "Tarde", "Noche"],
    include_lowest=True
)

print(df_preparado.dtypes)
df_preparado.head(15)

Fecha               datetime64[ns]
Hora                         int64
Sede                        object
CantidadIngresos             int64
DiaSemana                    int32
Mes                          int32
DiaMes                       int32
SemanaAnio                   int64
EsLunes                      int64
EsViernes                    int64
Turno                     category
dtype: object


,Fecha,Hora,Sede,CantidadIngresos,DiaSemana,Mes,DiaMes,SemanaAnio,EsLunes,EsViernes,Turno
0,2026-04-10,8,Central,0,4,4,10,15,0,1,Mañana
1,2026-04-10,9,Central,0,4,4,10,15,0,1,Mañana
2,2026-04-10,10,Central,0,4,4,10,15,0,1,Mañana
3,2026-04-10,11,Central,0,4,4,10,15,0,1,Mañana
4,2026-04-10,12,Central,2,4,4,10,15,0,1,Mañana
5,2026-04-10,13,Central,2,4,4,10,15,0,1,Tarde
6,2026-04-10,14,Central,0,4,4,10,15,0,1,Tarde
7,2026-04-10,15,Central,0,4,4,10,15,0,1,Tarde
8,2026-04-10,16,Central,0,4,4,10,15,0,1,Tarde
9,2026-04-10,17,Central,0,4,4,10,15,0,1,Tarde


In [13]:
print("RESUMEN DEL DATASET PREPARADO")
print("-" * 45)

print("Filas:", len(df_preparado))
print("Columnas:", len(df_preparado.columns))
print("Ingresos conservados:",
      df_preparado["CantidadIngresos"].sum())
print("Fecha inicial:",
      df_preparado["Fecha"].min())
print("Fecha final:",
      df_preparado["Fecha"].max())
print("Sedes:",
      df_preparado["Sede"].unique())
print("Horas:",
      sorted(df_preparado["Hora"].unique()))

print("\nValores nulos:")
print(df_preparado.isnull().sum())

RESUMEN DEL DATASET PREPARADO
---------------------------------------------
Filas: 1248
Columnas: 11
Ingresos conservados: 32779
Fecha inicial: 2026-04-10 00:00:00
Fecha final: 2026-09-28 00:00:00
Sedes: ['Central']
Horas: [np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20)]

Valores nulos:
Fecha               0
Hora                0
Sede                0
CantidadIngresos    0
DiaSemana           0
Mes                 0
DiaMes              0
SemanaAnio          0
EsLunes             0
EsViernes           0
Turno               0
dtype: int64


In [14]:
nombre_salida = "dataset_afluencia_real_preparado.csv"

df_preparado.to_csv(
    nombre_salida,
    index=False,
    encoding="utf-8-sig"
)

files.download(nombre_salida)

print("Archivo generado:", nombre_salida)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Archivo generado: dataset_afluencia_real_preparado.csv
